# Sofascore - descarga de calendarios

Notebook de trabajo para descargar en local los partidos de `1RFEF` y `2RFEF`, validar los campos devueltos por Sofascore y exportar una tabla limpia a CSV.

Objetivo de esta primera versión:
- probar que los endpoints devuelven los partidos esperados
- extraer `competicion`, `grupo`, `jornada`, `fecha`, `equipo_local`, `equipo_visitante`, `event_id`, `status`
- guardar el resultado en `data/raw/` para revisarlo antes de llevarlo a Google Sheets

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd

In [2]:
# Ajusta estas rutas si quieres exportar a otra carpeta.
PROJECT_ROOT = Path('/Users/macmontxinho/Desktop/Teams/Unionistas/scouting')
EXPORT_DIR = PROJECT_ROOT / 'data' / 'raw'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# IDs validados desde DevTools / Network de Sofascore.
SOFASCORE_COMPETITIONS = {
    '1RFEF': {
        'unique_tournament_id': 17073,
        'season_id': 77727,
        'groups': {
            'Group 1': 94190,
            'Group 2': 94191,
        },
    },
    '2RFEF': {
        'unique_tournament_id': 544,
        'season_id': 77733,
        'groups': {
            'Group I': 93714,
            'Group II': 93715,
            'Group III': 93716,
            'Group IV': 93717,
            'Group V': 93718,
        },
    },
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json, text/plain, */*',
}

API_TEMPLATE = 'https://www.sofascore.com/api/v1/unique-tournament/{unique_tournament_id}/season/{season_id}/events/round/{round_number}'


## 1. Probar una jornada concreta

Si este endpoint devuelve JSON con `events`, `homeTeam`, `awayTeam`, `roundInfo`, `startTimestamp` y `tournament.groupName`, ya tenemos la pieza correcta.

In [3]:
competition_key = '1RFEF'   # '1RFEF' o '2RFEF'
round_number = 29           # cambia según la jornada que quieras probar

config = SOFASCORE_COMPETITIONS[competition_key]
url = API_TEMPLATE.format(
    unique_tournament_id=config['unique_tournament_id'],
    season_id=config['season_id'],
    round_number=round_number,
)

response = requests.get(url, headers=HEADERS, timeout=20)
response.raise_for_status()
payload = response.json()

print('URL:', url)
print('Claves raíz:', list(payload.keys()))
print('Nº eventos:', len(payload.get('events', [])))

URL: https://www.sofascore.com/api/v1/unique-tournament/17073/season/77727/events/round/29
Claves raíz: ['events', 'hasNextPage']
Nº eventos: 20


In [4]:
payload.get('events', [])[0] if payload.get('events') else {}

{'eventState': {},
 'tournament': {'name': 'Primera Federacion, Group 1',
  'slug': 'primera-federacion-group-1',
  'category': {'name': 'Spain',
   'slug': 'spain',
   'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
   'country': {'alpha2': 'ES',
    'alpha3': 'ESP',
    'name': 'Spain',
    'slug': 'spain'},
   'id': 32,
   'flag': 'spain',
   'alpha2': 'ES',
   'fieldTranslations': {'nameTranslation': {'ar': 'إسبانيا',
     'hi': 'स्पेन',
     'bn': 'স্পেন',
     'ru': 'Испания'},
    'shortNameTranslation': {}}},
  'uniqueTournament': {'name': 'Primera Federación',
   'slug': 'primera-federacion',
   'primaryColorHex': '#fe0235',
   'secondaryColorHex': '#8c8c8c',
   'category': {'name': 'Spain',
    'slug': 'spain',
    'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
    'country': {'alpha2': 'ES',
     'alpha3': 'ESP',
     'name': 'Spain',
     'slug': 'spain'},
    'id': 32,
    'flag': 'spain',
    'alpha2': 'ES',
    'fieldTranslations': {'nameTranslat

## 2. Normalizar una jornada a tabla

In [5]:
def unix_to_datetime(value):
    if not value:
        return pd.NaT
    return datetime.fromtimestamp(value, tz=timezone.utc).astimezone()


def normalize_events(events, competition_label):
    rows = []
    for event in events:
        tournament = event.get('tournament', {}) or {}
        home = event.get('homeTeam', {}) or {}
        away = event.get('awayTeam', {}) or {}
        status = event.get('status', {}) or {}
        round_info = event.get('roundInfo', {}) or {}
        season = event.get('season', {}) or {}

        rows.append(
            {
                'competicion': competition_label,
                'grupo': tournament.get('groupName') or '',
                'tournament_name': tournament.get('name') or '',
                'unique_tournament_id': (tournament.get('uniqueTournament') or {}).get('id'),
                'group_tournament_id': tournament.get('id'),
                'season_id': season.get('id'),
                'season_name': season.get('name') or '',
                'jornada': round_info.get('round'),
                'event_id': event.get('id'),
                'slug': event.get('slug') or '',
                'fecha': unix_to_datetime(event.get('startTimestamp')),
                'timestamp': event.get('startTimestamp'),
                'status_code': status.get('code'),
                'status_type': status.get('type') or '',
                'status_desc': status.get('description') or '',
                'equipo_local': home.get('name') or '',
                'equipo_local_id': home.get('id'),
                'equipo_visitante': away.get('name') or '',
                'equipo_visitante_id': away.get('id'),
            }
        )
    return pd.DataFrame(rows)


events_df = normalize_events(payload.get('events', []), competition_key)
events_df.head()

,competicion,grupo,tournament_name,unique_tournament_id,group_tournament_id,season_id,season_name,jornada,event_id,slug,fecha,timestamp,status_code,status_type,status_desc,equipo_local,equipo_local_id,equipo_visitante,equipo_visitante_id
0,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,29,14102904,merida-ad-athletic-clubu21,2026-03-20 19:00:00+01:00,1774029600,0,notstarted,Not started,Athletic Club B U21,24324,Mérida AD,24351
1,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,29,14102902,cd-arenteiro-celta-vigo,2026-03-20 20:30:00+01:00,1774035000,0,notstarted,Not started,CD Arenteiro,263555,Celta Vigo B,24336
2,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,29,14103014,betis-deportivo-hercules-cf,2026-03-20 21:15:00+01:00,1774037700,0,notstarted,Not started,Betis Deportivo,24358,Hércules CF,2877
3,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,29,14102930,zamora-real-madrid-castilla,2026-03-21 14:00:00+01:00,1774098000,0,notstarted,Not started,Real Madrid Castilla U21,5069,Zamora CF,24379
4,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,29,14102940,ourense-cf-pontevedra-cf,2026-03-21 16:15:00+01:00,1774106100,0,notstarted,Not started,Pontevedra CF,4941,Ourense CF,263550


## 3. Descargar varias jornadas o una temporada completa

In [6]:
def fetch_round(competition_key: str, round_number: int) -> pd.DataFrame:
    config = SOFASCORE_COMPETITIONS[competition_key]
    url = API_TEMPLATE.format(
        unique_tournament_id=config['unique_tournament_id'],
        season_id=config['season_id'],
        round_number=round_number,
    )
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    payload = response.json()
    return normalize_events(payload.get('events', []), competition_key)


def fetch_season(competition_key: str, rounds: list[int]) -> pd.DataFrame:
    frames = []
    for round_number in rounds:
        round_df = fetch_round(competition_key, round_number)
        frames.append(round_df)
    if not frames:
        return pd.DataFrame()
    season_df = pd.concat(frames, ignore_index=True)
    season_df = season_df.sort_values(['jornada', 'fecha', 'grupo', 'equipo_local'])
    return season_df.drop_duplicates(subset=['event_id']).reset_index(drop=True)


In [7]:
# Ejemplo: descargar unas pocas jornadas para validar campos.
sample_rounds = [1, 2, 3]
sample_df = fetch_season('1RFEF', sample_rounds)
sample_df.head(20)

,competicion,grupo,tournament_name,unique_tournament_id,group_tournament_id,season_id,season_name,jornada,event_id,slug,fecha,timestamp,status_code,status_type,status_desc,equipo_local,equipo_local_id,equipo_visitante,equipo_visitante_id
0,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,1,14099027,cd-lugo-real-madrid-castilla,2025-08-29 19:15:00+02:00,1756487700,100,finished,Ended,Real Madrid Castilla U21,5069,CD Lugo,24332
1,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,1,14099042,algeciras-cf-gimnastic-de-tarragona,2025-08-29 21:30:00+02:00,1756495800,100,finished,Ended,Gimnàstic de Tarragona,2840,Algeciras CF,4489
2,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,1,14099032,pontevedra-cf-cp-cacereno,2025-08-30 17:00:00+02:00,1756566000,100,finished,Ended,Pontevedra CF,4941,CP Cacereño,2878
3,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,1,14099039,cd-eldense-ce-sabadell,2025-08-30 19:15:00+02:00,1756574100,100,finished,Ended,CE Sabadell,24335,CD Eldense,47321
4,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,1,14099030,merida-ad-barakaldo-cf,2025-08-30 19:30:00+02:00,1756575000,100,finished,Ended,Mérida AD,24351,Barakaldo CF,2879
5,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,1,14099031,unionistas-de-salamanca-cf-osasuna,2025-08-30 19:30:00+02:00,1756575000,100,finished,Ended,Unionistas de Salamanca CF,254151,Osasuna B,24328
6,1RFEF,Group 1,"Primera Federacion, Group 1",17073,94190,77727,Primera Federacion 25/26,1,14099028,zamora-celta-vigo,2025-08-30 19:30:00+02:00,1756575000,100,finished,Ended,Zamora CF,24379,Celta Vigo B,24336
7,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,1,14099041,juventud-torremolinos-cf-ce-europa,2025-08-30 19:30:00+02:00,1756575000,100,finished,Ended,Juventud Torremolinos CF,263407,CE Europa,135672
8,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,1,14099035,sociedad-deportivo-tarazona-hercules-cf,2025-08-30 21:30:00+02:00,1756582200,100,finished,Ended,Hércules CF,2877,SD Tarazona,254156
9,1RFEF,Group 2,"Primera Federacion, Group 2",17073,94191,77727,Primera Federacion 25/26,1,14099040,ud-ibiza-sevilla-atletico,2025-08-30 21:30:00+02:00,1756582200,100,finished,Ended,Sevilla Atlético,7762,UD Ibiza,24368


## 4. Exportar a CSV local

In [8]:
# Cambia la lista de jornadas según lo que quieras bajar.
rounds_1rfef = list(range(1, 39))
rounds_2rfef = list(range(1, 35))

# Descomenta cuando quieras exportar todo.
# calendario_1rfef = fetch_season('1RFEF', rounds_1rfef)
# calendario_2rfef = fetch_season('2RFEF', rounds_2rfef)
#
# calendario_1rfef.to_csv(EXPORT_DIR / 'sofascore_1rfef_calendario.csv', index=False)
# calendario_2rfef.to_csv(EXPORT_DIR / 'sofascore_2rfef_calendario.csv', index=False)
#
# print('Exportados:')
# print(EXPORT_DIR / 'sofascore_1rfef_calendario.csv')
# print(EXPORT_DIR / 'sofascore_2rfef_calendario.csv')

## 5. Comprobaciones útiles antes de llevarlo a la app

Revisa especialmente:
- si `grupo` viene bien en todos los partidos
- si `fecha` se convierte correctamente
- si `equipo_local` / `equipo_visitante` necesitan normalización para cruzar con campogramas
- si hay jornadas con partidos aplazados o sin fecha definitiva